# kernel_101：FlyDSL kernel 的组织方式

前面几个实验都在 kernel 里面看指令。这个实验看 kernel 外面的结构，回答这些问题：

1. `@flyc.kernel` 和 `@flyc.jit` 各管什么，trace 出来的 MLIR 长什么样；
2. 什么时候 trace、什么时候编译，缓存 key 由什么组成；
3. 为什么常见写法是一个带 `functools.lru_cache` 的 `build(...)` / `compile_xxx(...)` 函数，把编译期参数从外面传进去；
4. AST 改写做了什么，它让哪些 Python 写法行为变了；
5. `for` 循环有哪几种写法，各自变成什么 IR 和 ISA；
6. 子函数加不加 `@flyc.jit` 有什么区别；为什么 kernel 里可以调用 `@flyc.jit` 函数，却不能调用（launch）另一个 `@flyc.kernel`。

参考 [FlyDSL 仓库](https://github.com/ROCm/FlyDSL/tree/main) 的 kernel authoring guide；但行为以本机安装的 FlyDSL 0.3.4.1 为准（第 2 节的缓存行为和仓库 main 分支的文档不同）。ISA 部分同时编译 gfx942 / gfx950，GPU 结果在本机 gfx950 上检查。

## 0. 准备

- `dumps(launcher, *args, arch=...)`：只编译不运行，返回 `FLYDSL_DUMP_IR` 导出的各阶段文件；`00_origin.mlir` 是 trace 刚结束、还没跑任何 pass 的 IR；
- `clean(mlir)`：去掉 `loc(...)` 调试信息；`scf(mlir)`：只看 `scf.*` 控制流；
- `try_(fn)`：执行并打印异常类型和第一行信息；
- `launcher(k)`：用 1 个 wave 启动 kernel `k(X, Y)` 的 `@flyc.jit` launcher。

后面代码里 `# →` 开头的注释就是这一行的实测结果（GPU 结果和耗时来自本机 gfx950，ISA 统计同时编译 gfx942 / gfx950），cell 的输出只用来核对。

In [1]:
import os
os.environ["FLYDSL_RUNTIME_ENABLE_CACHE"] = "0"   # 关掉磁盘缓存；进程内的内存缓存仍然有效

import functools, glob, re, tempfile, time
from collections import Counter
from importlib.metadata import version

import torch
import flydsl.compiler as flyc
import flydsl.expr as fx

ARCHS = ("gfx942", "gfx950")
LOCAL_ARCH = torch.cuda.get_device_properties(0).gcnArchName.split(":")[0]
stream = fx.Stream(torch.cuda.current_stream().cuda_stream)
print("flydsl", version("flydsl"), "| torch", torch.__version__, "| 本机 GPU", LOCAL_ARCH)


def dumps(launcher, *args, arch="gfx942"):
    d = tempfile.mkdtemp(prefix="kernel101_")
    env = dict(ARCH=arch, COMPILE_ONLY="1", FLYDSL_DUMP_IR="1", FLYDSL_DUMP_DIR=d)
    old = {k: os.environ.get(k) for k in env}
    os.environ.update(env)
    try:
        flyc.compile(launcher, *args)
    finally:
        for k, v in old.items():
            if v is None:
                os.environ.pop(k)
            else:
                os.environ[k] = v
    return {os.path.basename(p): open(p).read() for p in glob.glob(f"{d}/**/*.*", recursive=True)}


def clean(mlir):
    lines = [re.sub(r"\s*loc\([^()]*(\([^()]*\))?[^()]*\)", "", l) for l in mlir.splitlines()]
    return "\n".join(l for l in lines if not l.startswith("#loc"))


def scf(mlir):
    return [l.strip() for l in clean(mlir).splitlines() if "scf." in l]


def isa_body(text):
    lines, inside = [], False
    for raw in text.splitlines():
        if re.match(r"^\w+:", raw):
            inside = True
            continue
        line = raw.split(";")[0].strip()
        if inside and line and not line.startswith("."):
            lines.append(line)
        if line.startswith("s_endpgm"):
            break
    return lines


def try_(fn):
    try:
        fn()
        print("    没有报错")
    except Exception as e:
        msg = str(e).strip().splitlines()
        detail = next((l.strip() for l in msg if "error:" in l), "")
        print(f"    {type(e).__name__}: {msg[0][:160]}" + (f"\n      {detail[:160]}" if detail else ""))


def launcher(k):
    @flyc.jit
    def launch(X: fx.Tensor, Y: fx.Tensor, stream: fx.Stream):
        k(X, Y).launch(grid=(1, 1, 1), block=(64, 1, 1), stream=stream)
    return launch


X = torch.randn(64, device="cuda")
Y = torch.zeros(64, device="cuda")

flydsl 0.3.4.1 | torch 2.9.1+rocm7.2.0.git7e1940d4 | 本机 GPU gfx950


## 1. 两层结构：`@flyc.kernel` 和 `@flyc.jit`

一个 kernel 带一个编译期参数 `k`，launcher 里用 `k=2`、`k=3` 各 launch 一次：

In [2]:
@flyc.kernel
def scale(X: fx.Tensor, Y: fx.Tensor, k: fx.Constexpr[int]):   # → k 不出现在 gpu.func 的参数列表里
    i = fx.block_idx.x * 64 + fx.thread_idx.x
    Y[i] = X[i] * float(k)                                      # → arith.mulf %x, 2.0（k=2 时）


def make_scale_twice():
    # 每次新建 launcher：看 gfx942 的 IR 和在本机 GPU 上运行要用两个不同的 launcher（原因见第 3 节末尾）
    @flyc.jit
    def scale_twice(X: fx.Tensor, Y: fx.Tensor, stream: fx.Stream):          # → host 端 func.func @scale_twice
        scale(X, Y, 2).launch(grid=(1, 1, 1), block=(64, 1, 1), stream=stream)  # → gpu.func @scale_0 + gpu.launch_func
        scale(Y, Y, 3).launch(grid=(1, 1, 1), block=(64, 1, 1), stream=stream)  # → gpu.func @scale_1 + gpu.launch_func
    return scale_twice


ir_ = clean(dumps(make_scale_twice(), X, Y, stream)["00_origin.mlir"])
for l in ir_.splitlines():                     # 只打印函数定义、常数和 launch
    if re.search(r"gpu\.func|func\.func|arith\.constant \d\.|launch_func", l):
        print(l.strip()[:110])
make_scale_twice()(X, Y, stream)
torch.cuda.synchronize()
print("GPU 结果正确:", torch.allclose(Y, X * 6))  # → True

[flydsl.compile] FLYDSL_DUMP_IR=1 dir=/tmp/kernel101_lukk6f9u/scale_twice
[flydsl.compile] dump 00_origin -> /tmp/kernel101_lukk6f9u/scale_twice/00_origin.mlir
[flydsl.compile] dump 01_fly_rewrite_func_signature -> /tmp/kernel101_lukk6f9u/scale_twice/01_fly_rewrite_func_signature.mlir
[flydsl.compile] dump 02_fly_canonicalize -> /tmp/kernel101_lukk6f9u/scale_twice/02_fly_canonicalize.mlir
[flydsl.compile] dump 03_fly_rocdl_expand_ops -> /tmp/kernel101_lukk6f9u/scale_twice/03_fly_rocdl_expand_ops.mlir
[flydsl.compile] dump 04_fly_layout_lowering -> /tmp/kernel101_lukk6f9u/scale_twice/04_fly_layout_lowering.mlir
[flydsl.compile] dump 05_fly_int_swizzle_simplify -> /tmp/kernel101_lukk6f9u/scale_twice/05_fly_int_swizzle_simplify.mlir
[flydsl.compile] dump 06_canonicalize -> /tmp/kernel101_lukk6f9u/scale_twice/06_canonicalize.mlir
[flydsl.compile] dump 07_fly_convert_atom_call_to_ssa_form -> /tmp/kernel101_lukk6f9u/scale_twice/07_fly_convert_atom_call_to_ssa_form.mlir
[flydsl.compile] dump 

一个 MLIR module 里有两部分：

- `gpu.module @kernels` 里的 `gpu.func`：每个 `@flyc.kernel` 的一次**特化**。`Constexpr` 参数不出现在参数列表里，直接变成常数（`2.0`、`3.0`），所以 `k=2` 和 `k=3` 是两个 kernel：`scale_0`、`scale_1`。
- host 端的 `func.func @scale_twice`：`@flyc.jit` 函数本身。它负责接收 torch tensor、算 grid、发出 `gpu.launch_func`。它在 host 上运行，编译后变成一个 C 函数。

所以 `@flyc.jit` 是 host launcher，`@flyc.kernel` 是 device 函数；kernel 只能在 jit 函数的 trace 过程中被调用和 `.launch()`。直接在 Python 里调用 kernel：

In [3]:
try_(lambda: scale(X, Y, 2))   # → RuntimeError: @kernel can only be called inside @jit function

    RuntimeError: @kernel can only be called inside @jit function


## 2. 什么时候 trace，什么时候编译

`@flyc.jit` 函数第一次被调用时：AST 改写 → trace（执行一遍 Python，生成上面的 MLIR）→ 跑 pass、生成二进制 → 放进缓存 → launch。之后再调用，先算缓存 key，命中就直接 launch，**不再执行 Python 函数体**。

用 trace 时的 `print` 来观察（`print` 只在 trace 时执行；GPU 运行时打印要用 `fx.printf`）：

In [4]:
@flyc.kernel
def scale_k(X: fx.Tensor, Y: fx.Tensor, k: fx.Constexpr[int]):
    i = fx.thread_idx.x
    Y[i] = X[i] * float(k)


@flyc.jit
def launch_k(X: fx.Tensor, Y: fx.Tensor, k: fx.Constexpr[int], stream: fx.Stream):
    print("    [trace]", X.type, "k =", k)
    scale_k(X, Y, k).launch(grid=(1, 1, 1), block=(64, 1, 1), stream=stream)


def timed(name, *args):
    t0 = time.perf_counter()
    launch_k(*args)
    torch.cuda.synchronize()
    print(f"{name:28} {(time.perf_counter() - t0) * 1e3:8.2f} ms")


X2 = torch.randn(128, device="cuda")
timed("第一次 (f32, k=2)", X, Y, 2, stream)                          # → trace + 编译，约 35 ms
timed("再调用一次", X, Y, 2, stream)                                  # → 命中缓存，不打印 [trace]，约 0.2 ms
timed("k=3", X, Y, 3, stream)                                         # → Constexpr 值变了：重新 trace + 编译
timed("换 shape：128 个元素", X2, torch.zeros_like(X2), 2, stream)    # → shape 是运行时值：命中缓存
timed("换 dtype：bf16", X.bfloat16(), Y.bfloat16(), 2, stream)        # → 重新 trace + 编译
timed("换 rank：8x8", X.view(8, 8), Y.view(8, 8), 2, stream)          # → 重新 trace + 编译

    [trace] !fly.memref<f32, global, ?:1> k = 2
第一次 (f32, k=2)                  35.05 ms
再调用一次                            0.19 ms
    [trace] !fly.memref<f32, global, ?:1> k = 3
k=3                             32.19 ms
换 shape：128 个元素                  0.17 ms
    [trace] !fly.memref<bf16, global, ?:1> k = 2
换 dtype：bf16                    31.67 ms
    [trace] !fly.memref<f32, global, (?,?):(?{i64},1)> k = 2
换 rank：8x8                      31.83 ms


- 缓存 key 由这些组成（`JitFunction._build_full_cache_key`）：
  - 每个参数的**签名**：tensor 的 dtype、rank、哪些 stride 是静态的（torch 传入时 shape 和 stride 是动态的 `?`，只有 stride=1 的维是静态 `1`），`Constexpr` 的值；
  - 编译目标、相关环境变量；
  - jit 函数（以及它调用的 kernel）用到的**全局变量**的快照。
- 注意：仓库 main 分支的文档说"每次调用都重新 trace，再按生成的 MLIR 查缓存"，那是更新的版本；本机 0.3.4.1 命中缓存时不 trace。

### 2.1 `flyc.compile`：跳过缓存查找

即使命中缓存，每次调用 jit 函数也要绑定参数、算 key。`flyc.compile(launch, *args)` 返回一个 `CompiledFunction`，之后直接调用它只做参数打包：

In [5]:
compiled = flyc.compile(launch_k, X, Y, 2, stream)
for name, fn in [("launch_k(...)", lambda: launch_k(X, Y, 2, stream)),    # → 约 35 us/次
                 ("compiled(...)", lambda: compiled(X, Y, 2, stream))]:   # → 约 4 us/次
    fn()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(1000):
        fn()
    torch.cuda.synchronize()
    print(f"{name:16} 每次调用 {(time.perf_counter() - t0) * 1e3:.1f} us（1000 次平均，含 kernel 执行）")

launch_k(...)    每次调用 34.8 us（1000 次平均，含 kernel 执行）
compiled(...)    每次调用 4.1 us（1000 次平均，含 kernel 执行）


benchmark 循环、CUDA graph 捕获这类反复 launch 同一个特化的场合，应该拿着 `CompiledFunction` 调用。

### 2.2 全局变量：编译后不能改

kernel 里用到的全局变量在第一次编译时被记下来，之后如果它变了，FlyDSL 直接报错，而不是悄悄用旧值或重新编译：

In [6]:
SCALE = 2.0


@flyc.kernel
def scale_global(X: fx.Tensor, Y: fx.Tensor):
    i = fx.thread_idx.x
    Y[i] = X[i] * SCALE


launch_global = launcher(scale_global)
launch_global(X, Y, stream)
torch.cuda.synchronize()
print("SCALE = 2.0 时 Y/X =", (Y[0] / X[0]).item())   # → 2.0

SCALE = 3.0
try_(lambda: launch_global(X, Y, stream))   # → RuntimeError: global 'SCALE' ... changed since first compile

SCALE = 2.0 时 Y/X = 2.0
    RuntimeError: FlyDSL: global 'SCALE' (module '__main__') used by @flyc.jit 'launch' changed since first compile (old=('scalar', 2.0), new=('scalar', 3.0)). Capturing live glo


这同样适用于 jit 函数体里被修改的全局变量（比如一个计数器列表）。报错信息建议把这个值作为显式参数传入，让它进入缓存 key。

## 3. 为什么用带缓存的 `build(...)` 函数

kernel 往往有很多编译期参数：tile 大小、dtype、layout、是否 swizzle……。可以选的传法：

| 传法 | 问题 |
|---|---|
| 全局变量 | 编译后不能改（2.2 节），不同配置没法共存 |
| `fx.Constexpr[...]` 参数 | 要从 jit 一层层传到 kernel 和每个子函数；只支持简单的可哈希值 |
| **外层 Python 函数的参数（闭包）** | 每组参数对应一组新的 kernel / jit 对象，互不干扰 |

所以常见写法是：

```python
@functools.lru_cache
def build(BM, BN, dtype, ...):        # 所有编译期参数
    @flyc.kernel
    def kern(...): ...                # 闭包里直接用 BM、BN（相当于 Python 常数）
    @flyc.jit
    def launch(A, B, C, stream): kern(...).launch(...)
    return launch
```

本仓库 [v0.py](../kernels/gemm/v0.py) 的 `build_v0(BM, BN, OCCUPANCY, M, N, K)` 就是这种结构（没有加缓存，调用方只 build 一次）。

为什么要加 `lru_cache`：每次调用 `build(...)` 都会新建 `KernelFunction` / `JitFunction` 对象，它们的内存缓存是空的，要重新 AST 改写、trace、编译。比较同一个配置 build 两次：

In [7]:
def build(scale):
    @flyc.kernel
    def kern(X: fx.Tensor, Y: fx.Tensor):
        i = fx.thread_idx.x
        Y[i] = X[i] * scale              # scale 是闭包里的 Python float，trace 时变成常数

    @flyc.jit
    def launch(X: fx.Tensor, Y: fx.Tensor, stream: fx.Stream):
        kern(X, Y).launch(grid=(1, 1, 1), block=(64, 1, 1), stream=stream)

    return launch


def timed_build(name, b, s):
    t0 = time.perf_counter()
    b(s)(X, Y, stream)
    torch.cuda.synchronize()
    print(f"{name:18} scale={s}: {(time.perf_counter() - t0) * 1e3:7.2f} ms，Y/X = {(Y[0] / X[0]).item():.4f}")


build_cached = functools.lru_cache(build)
timed_build("build", build, 2.0)                    # → 约 35 ms：trace + 编译
timed_build("build", build, 2.0)                    # → 约 33 ms：参数相同，仍然重新编译
timed_build("build", build, 3.0)                    # → 约 33 ms
timed_build("lru_cache(build)", build_cached, 2.0)  # → 约 32 ms：第一次
timed_build("lru_cache(build)", build_cached, 2.0)  # → 约 0.2 ms：拿到同一个 launcher，命中缓存
timed_build("lru_cache(build)", build_cached, 3.0)  # → 约 30 ms：新配置，单独编译，Y/X = 3

build              scale=2.0:   36.60 ms，Y/X = 2.0000
build              scale=2.0:   35.57 ms，Y/X = 2.0000
build              scale=3.0:   33.22 ms，Y/X = 3.0000
lru_cache(build)   scale=2.0:   28.42 ms，Y/X = 2.0000
lru_cache(build)   scale=2.0:    0.14 ms，Y/X = 2.0000
lru_cache(build)   scale=3.0:   27.74 ms，Y/X = 3.0000


- 这里的 kernel 很小，重新编译只要几十毫秒；GEMM 这类大 kernel 要几秒。
- 不同配置（`scale=3.0`）各自有自己的 launcher，互不影响，也不会触发 2.2 节的全局变量报错。

一个相关的坑：编译目标（gfx942 / gfx950）在 launcher **第一次被调用时**确定，之后固定。同一个 launcher 先用 `ARCH=gfx942` 编译、再在本机 gfx950 上运行，会拿到 gfx942 的二进制，运行时报 HIP 错误（本机看到的是 `hipErrorInvalidHandle`）。所以这几个 notebook 里看 ISA 和在 GPU 上运行时，总是分别 build 新的 launcher。

## 4. AST 改写

`@flyc.kernel` / `@flyc.jit` 在装饰时用 `inspect.getsource` 取出函数源码，改写 AST 后重新编译成 Python 函数。改写只作用于这个函数自己的源码（包括写在它里面的嵌套函数），不影响它调用的外部函数。

用 FlyDSL 自己的改写器处理下面这段代码，看改写后的 Python：

In [8]:
import ast, inspect, textwrap, types
from flydsl.compiler.ast_rewriter import ASTRewriter


def demo(X, n):
    acc = X[0]
    for i in fx.range_constexpr(2):   # → 普通 Python for
        acc = acc + 1.0
    for i in range(n):                # → __for_body_0 + scf_for_dispatch，acc 是 loop-carried 值
        acc = acc * 2.0
    if acc > 0.0:                     # → __then_0 / __else_0 + scf_if_dispatch
        acc = -acc
    y = acc if fx.const_expr(n > 4) else acc + 1.0   # → scf_ifexp_dispatch
    return y


module = ast.parse(textwrap.dedent(inspect.getsource(demo)))
context = types.SimpleNamespace(python_globals=demo.__globals__, filename="<demo>")
print("改写器:", [t.__name__ for t in ASTRewriter.transformers])
for t in ASTRewriter.transformers:
    module.body[0] = t(context=context, first_lineno=0).visit(module.body[0])
print(ast.unparse(module))

改写器: ['RewriteBoolOps', 'ReplaceIfWithDispatch', 'InsertEmptyYieldForSCFFor', 'ReplaceYieldWithSCFYield', 'CanonicalizeWhile', 'FallbackLocations']
def demo(X, n):
    acc = X[0]
    for i in range(2):
        acc = acc + 1.0

    def __for_body_0(i, __ret_names, acc):
        acc = acc * 2.0
        return scf_for_collect_results(__ret_names, locals())
    acc = scf_for_dispatch(0, n, 1, __for_body_0, result_names=('acc',), result_values=(__check_local_var('acc', locals()),))

    def __then_0(__ret_names, acc):
        acc = -acc
        return scf_if_collect_results(__ret_names, locals())

    def __else_0(__ret_names, acc):
        return scf_if_collect_results(__ret_names, locals())
    acc = scf_if_dispatch(acc > 0.0, __then_0, __else_0, result_names=('acc',), result_values=(__check_local_var('acc', locals()),), auto_else=True)
    y = scf_ifexp_dispatch(fx.const_expr(n > 4), lambda : acc, lambda : acc + 1.0)
    return y


| 原写法 | 改写成 | trace 时 |
|---|---|---|
| `for i in range_constexpr(n)` | 普通 `for i in range(n)` | Python 循环，循环体展开 n 次 |
| `for i in range(...)` | 循环体变成函数 `__for_body_N`，交给 `scf_for_dispatch` | 生成 `scf.for`；循环体里赋值、且循环前已存在的变量（这里的 `acc`）自动成为 loop-carried 值 |
| `if cond:` | 两个分支变成 `__then_N` / `__else_N`，交给 `scf_if_dispatch` | `cond` 是运行时值 → `scf.if`；是 Python 值 → 直接执行选中的分支 |
| `a if c else b` | `scf_ifexp_dispatch(c, lambda: a, lambda: b)` | 同上 |
| `and` / `or` / `not` | `dsl_and_` / `dsl_or_` / `dsl_not_` | 运行时值用位运算，Python 值保持原语义 |
| `while cond:` | `CanonicalizeWhile` 改写 | 运行时条件生成 `scf.while` |
| `if fx.const_expr(c):` | 去掉包装，保留普通 `if` | 纯 Python 分支 |

`if` 和 `for` 的分支 / 循环体都变成了**嵌套函数**，变量通过参数传进去、通过返回值带出来。这带来几个和普通 Python 不同的地方：

In [9]:
MODE = True


@flyc.kernel
def if_existing(X: fx.Tensor, Y: fx.Tensor):     # 变量在 if 之前已存在
    i = fx.thread_idx.x
    v = X[i]
    if v > 0.0:                                  # → scf.if -> (f32)，两个分支各 scf.yield 一个值
        v = v * 2.0
    Y[i] = v


@flyc.kernel
def if_new_var(X: fx.Tensor, Y: fx.Tensor):      # 变量只在分支里定义
    i = fx.thread_idx.x
    if X[i] > 0.0:
        w = X[i] * 2.0
    else:
        w = X[i]
    Y[i] = w                                     # → NameError: name 'w' is not defined


@flyc.kernel
def if_python_bool(X: fx.Tensor, Y: fx.Tensor):  # 条件是 Python 值，但没用 const_expr
    i = fx.thread_idx.x
    if MODE:
        w = X[i] * 2.0
    Y[i] = w                                     # → 同样 NameError


@flyc.kernel
def if_const_expr(X: fx.Tensor, Y: fx.Tensor):   # 用 const_expr
    i = fx.thread_idx.x
    if fx.const_expr(MODE):                      # → 普通 Python if，没有 scf.if，没有报错
        w = X[i] * 2.0
    Y[i] = w


@flyc.kernel
def if_none_init(X: fx.Tensor, Y: fx.Tensor):    # 先赋值为 None
    i = fx.thread_idx.x
    w = None
    if X[i] > 0.0:                               # → TypeError: ['w'] initialized as None before a dynamic if/else
        w = X[i]
    else:
        w = -X[i]
    Y[i] = w


for k in [if_existing, if_new_var, if_python_bool, if_const_expr, if_none_init]:
    print(f"{k._func.__name__}:")
    try_(lambda k=k: print("    scf:", scf(dumps(launcher(k), X, Y, stream)["00_origin.mlir"])))

if_existing:
[flydsl.compile] FLYDSL_DUMP_IR=1 dir=/tmp/kernel101_wu91kfnn/if_existing_0
[flydsl.compile] dump 00_origin -> /tmp/kernel101_wu91kfnn/if_existing_0/00_origin.mlir
[flydsl.compile] dump 01_fly_rewrite_func_signature -> /tmp/kernel101_wu91kfnn/if_existing_0/01_fly_rewrite_func_signature.mlir
[flydsl.compile] dump 02_fly_canonicalize -> /tmp/kernel101_wu91kfnn/if_existing_0/02_fly_canonicalize.mlir
[flydsl.compile] dump 03_fly_rocdl_expand_ops -> /tmp/kernel101_wu91kfnn/if_existing_0/03_fly_rocdl_expand_ops.mlir
[flydsl.compile] dump 04_fly_layout_lowering -> /tmp/kernel101_wu91kfnn/if_existing_0/04_fly_layout_lowering.mlir
[flydsl.compile] dump 05_fly_int_swizzle_simplify -> /tmp/kernel101_wu91kfnn/if_existing_0/05_fly_int_swizzle_simplify.mlir
[flydsl.compile] dump 06_canonicalize -> /tmp/kernel101_wu91kfnn/if_existing_0/06_canonicalize.mlir
[flydsl.compile] dump 07_fly_convert_atom_call_to_ssa_form -> /tmp/kernel101_wu91kfnn/if_existing_0/07_fly_convert_atom_call_to_ssa_f

- 分支里新定义的变量不会被带出分支函数，要先在 `if` 之前给它一个同类型的初值。条件是 Python 值时也一样（trace 时只执行 then 分支，变量照样带不出来），这就是 [LDS_101](lds_101.ipynb) 里遇到的问题；用 `fx.const_expr` 包住条件才保留原来的 `if`。
- `None` 不是 MLIR 值，不能作为 `scf.if` 的结果，报错信息直接给出了改法。

还有两个只能靠自己注意的地方：

In [10]:
@flyc.kernel
def early_return(X: fx.Tensor, Y: fx.Tensor):
    i = fx.thread_idx.x
    if X[i] > 0.0:
        return                 # 想让正数的 lane 不写 Y → 被静默忽略
    Y[i] = X[i]                # → 所有 lane 都执行了


Y.fill_(7.0)
launcher(early_return)(X, Y, stream)
torch.cuda.synchronize()
print("early return：正数 lane 的 Y 仍是 7？", bool((Y[X > 0] == 7).all()), "| 所有 lane 都写了 X？", torch.equal(Y, X))   # → False | True

src = """
@flyc.kernel
def with_break(X: fx.Tensor, Y: fx.Tensor):
    acc = X[0]
    for j in range(4):
        if acc > 10.0:
            break              # → 装饰时 SyntaxError: 'break' outside loop
        acc = acc * 2.0
    Y[0] = acc
"""
path = os.path.join(tempfile.mkdtemp(), "with_break.py")
with open(path, "w") as f:
    f.write("import flydsl.compiler as flyc\nimport flydsl.expr as fx\n" + src)
import importlib.util
spec = importlib.util.spec_from_file_location("with_break", path)
try_(lambda: spec.loader.exec_module(importlib.util.module_from_spec(spec)))

early return：正数 lane 的 Y 仍是 7？ False | 所有 lane 都写了 X？ True
    SyntaxError: 'break' outside loop (with_break.py, line 9)


- **运行时 `if` 里的 `return` 被静默忽略**：它只是从生成的 `__then_N` 函数里返回，`if` 之后的代码对所有 lane 都执行了，没有任何报错。要表达"部分 lane 不写"，把写操作放进 `if` 里（或者对条件取反）。
- **循环体里的 `break` / `continue`**：循环体变成了函数，`break` 不在循环里，装饰时（`@flyc.kernel` 那一行）就报 `SyntaxError`。

## 5. `for` 循环的几种写法

同一个计算 `acc = acc * 1.5 + j`（j = 0..3），五种写法。注释里的 ISA 统计 gfx942 和 gfx950 相同：

In [11]:
@flyc.kernel
def loop_constexpr(X: fx.Tensor, Y: fx.Tensor, n: fx.Int32):
    i = fx.thread_idx.x
    acc = X[i]
    # → trace 时展开，没有 scf；ISA：8 条乘加，无跳转
    for j in fx.range_constexpr(4):
        acc = acc * 1.5 + float(j)
    Y[i] = acc


@flyc.kernel
def loop_range_const(X: fx.Tensor, Y: fx.Tensor, n: fx.Int32):
    i = fx.thread_idx.x
    acc = X[i]
    # → scf.for 常数边界 0 to 4；LLVM 完全展开，ISA 同上：8 条乘加，无跳转
    for j in range(4):
        acc = acc * 1.5 + fx.Float32(fx.Int32(j))
    Y[i] = acc


@flyc.kernel
def loop_range_runtime(X: fx.Tensor, Y: fx.Tensor, n: fx.Int32):
    i = fx.thread_idx.x
    acc = X[i]
    # → scf.for 运行时边界；LLVM 按 4 展开 + 余数循环：18 条乘加，6 个条件跳转
    for j in range(n):
        acc = acc * 1.5 + fx.Float32(fx.Int32(j))
    Y[i] = acc


@flyc.kernel
def loop_unroll2(X: fx.Tensor, Y: fx.Tensor, n: fx.Int32):
    i = fx.thread_idx.x
    acc = X[i]
    # → 同上，带 loop_annotation；只按 2 展开：6 条乘加，5 个条件跳转
    for j in range(fx.Int32(0), n, fx.Int32(1), unroll=2):
        acc = acc * 1.5 + fx.Float32(fx.Int32(j))
    Y[i] = acc


@flyc.kernel
def loop_init_yield(X: fx.Tensor, Y: fx.Tensor, n: fx.Int32):
    i = fx.thread_idx.x
    # → scf.for ... iter_args(...)；ISA 同 range(n)：18 条乘加，6 个条件跳转（v1/v2 用的写法）
    for j, state in range(fx.Int32(0), n, fx.Int32(1), init=[X[i]]):
        acc = state[0] * 1.5 + fx.Float32(fx.Int32(j))
        results = yield [acc]
    Y[i] = results


def loop_launcher(k):
    @flyc.jit
    def launch(X: fx.Tensor, Y: fx.Tensor, n: fx.Int32, stream: fx.Stream):
        k(X, Y, n).launch(grid=(1, 1, 1), block=(64, 1, 1), stream=stream)
    return launch


expect = X.clone()
for j in range(4):
    expect = expect * 1.5 + j

for k in [loop_constexpr, loop_range_const, loop_range_runtime, loop_unroll2, loop_init_yield]:
    Y.zero_()
    loop_launcher(k)(X, Y, 4, stream)
    torch.cuda.synchronize()
    files = {arch: dumps(loop_launcher(k), X, Y, 4, stream, arch=arch) for arch in ARCHS}
    stats = []
    for arch in ARCHS:
        body = isa_body(files[arch]["22_final_isa.s"])
        n_math = sum(bool(re.match(r"v_(fma|fmac|mul|add)_f32", l)) for l in body)
        n_branch = sum(l.startswith("s_cbranch") for l in body)
        stats.append(f"{arch} {n_math:2} 乘加 / {n_branch} 跳转")
    has_for = "有" if any("scf.for" in l for l in scf(files["gfx942"]["00_origin.mlir"])) else "无"
    print(f"{k._func.__name__:20} GPU 正确 {torch.allclose(Y, expect, rtol=1e-5)} | scf.for {has_for} | " + " | ".join(stats))

[flydsl.compile] FLYDSL_DUMP_IR=1 dir=/tmp/kernel101_hsswxr6c/loop_constexpr_0
[flydsl.compile] dump 00_origin -> /tmp/kernel101_hsswxr6c/loop_constexpr_0/00_origin.mlir
[flydsl.compile] dump 01_fly_rewrite_func_signature -> /tmp/kernel101_hsswxr6c/loop_constexpr_0/01_fly_rewrite_func_signature.mlir
[flydsl.compile] dump 02_fly_canonicalize -> /tmp/kernel101_hsswxr6c/loop_constexpr_0/02_fly_canonicalize.mlir
[flydsl.compile] dump 03_fly_rocdl_expand_ops -> /tmp/kernel101_hsswxr6c/loop_constexpr_0/03_fly_rocdl_expand_ops.mlir
[flydsl.compile] dump 04_fly_layout_lowering -> /tmp/kernel101_hsswxr6c/loop_constexpr_0/04_fly_layout_lowering.mlir
[flydsl.compile] dump 05_fly_int_swizzle_simplify -> /tmp/kernel101_hsswxr6c/loop_constexpr_0/05_fly_int_swizzle_simplify.mlir
[flydsl.compile] dump 06_canonicalize -> /tmp/kernel101_hsswxr6c/loop_constexpr_0/06_canonicalize.mlir
[flydsl.compile] dump 07_fly_convert_atom_call_to_ssa_form -> /tmp/kernel101_hsswxr6c/loop_constexpr_0/07_fly_convert_atom

几个细节：

- 边界要求是 i32：Python `int` 和 `fx.Int32` 都可以；`index` 类型会给警告并转换，i64 报错。
- 自动推断的 `range(...)` 里，循环变量 `j` 是裸的 i32 值，要用 `fx.Int32(j)` 包一下才能参与 `fx.Float32(...)` 这类转换；显式 `init` 写法里 `j` 是 `index` 类型，同样用 `fx.Int32(j)`。
- 显式写法 `for j, state in range(..., init=[...])`：`state` 是本轮的 loop-carried 值列表，`yield [新值]` 结束一轮，`results = yield ...` 在循环结束后得到最终值（只有一个值时直接是那个值，不是列表）。
- 显式写法的好处是 loop-carried 值完全由自己指定；自动推断会把循环体里赋值、且循环前已存在的变量都当作 loop-carried 值，变量多时容易带上不需要的值。

## 6. 子函数：加不加 `@flyc.jit`

同一段逻辑"正数乘 2"，写成三种子函数，在 kernel 里调用：

In [12]:
def plain_helper(v):                 # 普通 Python 函数，不经过 AST 改写
    if v > 0.0:
        v = v * 2.0
    return v


@flyc.jit
def jit_helper(v):                   # 加了 @flyc.jit：经过 AST 改写
    if v > 0.0:
        v = v * 2.0
    return v


def plain_unroll(v, times):          # 普通 Python 函数，只用 Python 控制流
    for _ in range(times):
        v = v + 1.0
    return v


@flyc.kernel
def use_plain(X: fx.Tensor, Y: fx.Tensor):
    i = fx.thread_idx.x
    Y[i] = plain_helper(X[i])        # → RuntimeError: cannot evaluate dynamic 'Boolean' as Python bool


@flyc.kernel
def use_jit(X: fx.Tensor, Y: fx.Tensor):
    i = fx.thread_idx.x
    Y[i] = jit_helper(X[i])          # → 内联（没有 func.call），生成 scf.if，结果正确


@flyc.kernel
def use_nested(X: fx.Tensor, Y: fx.Tensor):
    i = fx.thread_idx.x

    def nested(v):                   # 写在 kernel 里面：随 kernel 一起被改写
        if v > 0.0:
            v = v * 2.0
        return v

    Y[i] = nested(X[i])              # → 同 @flyc.jit：scf.if，结果正确


@flyc.kernel
def use_plain_unroll(X: fx.Tensor, Y: fx.Tensor):
    i = fx.thread_idx.x
    Y[i] = plain_unroll(X[i], 3)     # → 相当于宏：3 条 arith.addf


expect = torch.where(X > 0, X * 2, X)
for k in [use_plain, use_jit, use_nested, use_plain_unroll]:
    print(f"{k._func.__name__}:")

    def go(k=k):
        ir_ = clean(dumps(launcher(k), X, Y, stream)["00_origin.mlir"])
        print("    IR:", [l.strip()[:70] for l in ir_.splitlines() if re.search(r"scf\.if|func\.call|arith\.(mulf|addf)", l)])
        Y.zero_()
        launcher(k)(X, Y, stream)
        torch.cuda.synchronize()
        if k is not use_plain_unroll:
            print("    GPU 正确:", torch.equal(Y, expect))

    try_(go)

use_plain:
    RuntimeError: cannot evaluate dynamic 'Boolean' as Python bool during tracing
use_jit:
[flydsl.compile] FLYDSL_DUMP_IR=1 dir=/tmp/kernel101_qpqkrtp0/use_jit_0
[flydsl.compile] dump 00_origin -> /tmp/kernel101_qpqkrtp0/use_jit_0/00_origin.mlir
[flydsl.compile] dump 01_fly_rewrite_func_signature -> /tmp/kernel101_qpqkrtp0/use_jit_0/01_fly_rewrite_func_signature.mlir
[flydsl.compile] dump 02_fly_canonicalize -> /tmp/kernel101_qpqkrtp0/use_jit_0/02_fly_canonicalize.mlir
[flydsl.compile] dump 03_fly_rocdl_expand_ops -> /tmp/kernel101_qpqkrtp0/use_jit_0/03_fly_rocdl_expand_ops.mlir
[flydsl.compile] dump 04_fly_layout_lowering -> /tmp/kernel101_qpqkrtp0/use_jit_0/04_fly_layout_lowering.mlir
[flydsl.compile] dump 05_fly_int_swizzle_simplify -> /tmp/kernel101_qpqkrtp0/use_jit_0/05_fly_int_swizzle_simplify.mlir
[flydsl.compile] dump 06_canonicalize -> /tmp/kernel101_qpqkrtp0/use_jit_0/06_canonicalize.mlir
[flydsl.compile] dump 07_fly_convert_atom_call_to_ssa_form -> /tmp/kernel101

- `plain_helper` 不经过 AST 改写，`if v > 0.0` 是 Python 的 `if`，要把运行时值当 `bool`，所以报错。
- 写在 kernel 函数体里的嵌套函数是 kernel 源码的一部分，也被改写，效果和 `@flyc.jit` 一样。
- 只用编译期控制流的普通函数没问题，trace 时直接执行。v1 的 `lds_chunk` 这类只做地址计算的小函数就是这种用法。

所以规则是：**子函数里有对运行时值的 `if` / `for` / `while` 时，加 `@flyc.jit`（或写在 kernel 里面）；只有编译期逻辑时，普通函数即可。**

### 6.1 为什么 kernel 能调用 jit，却不能 launch kernel

看两个 `__call__`（FlyDSL 0.3.4.1）：

```python
# JitFunction.__call__（flydsl/compiler/jit_function.py）
def __call__(self, *args, **kwargs):
    if ir.Context.current is not None:      # 已经在某个 trace 里面
        return self.func(*args, **kwargs)   # 直接调用改写后的 Python 函数：内联
    ...                                     # 否则：作为 host launcher 编译并运行

# KernelFunction.__call__（flydsl/compiler/kernel_function.py）
def __call__(self, *args, **kwargs):
    ctx = CompilationContext.get_current()  # 只在 trace 一个 jit launcher 时存在
    if ctx is None:
        raise RuntimeError("@kernel can only be called inside @jit function")
    ...                                     # 返回 KernelLauncher，.launch() 发出 gpu.launch_func
```

- 在 kernel 里调用 `@flyc.jit` 函数时，trace 已经在进行，所以它只是一个被 AST 改写过的普通函数调用，生成的指令直接插在当前位置。
- `@flyc.kernel` 的 `.launch()` 在**当前插入点**发出 `gpu.launch_func`。在 jit 函数里，插入点是 host 函数；在 kernel 里，插入点在 `gpu.func` 内部，相当于在 GPU 上启动另一个 kernel，后续 pass 无法 lower：

In [13]:
@flyc.kernel
def inner(X: fx.Tensor, Y: fx.Tensor):
    i = fx.thread_idx.x
    Y[i] = X[i]


@flyc.kernel
def outer_launch(X: fx.Tensor, Y: fx.Tensor):
    # → gpu.launch_func 出现在 gpu.func 里，编译报错 failed to legalize operation 'gpu.launch_func'
    inner(X, Y).launch(grid=(1, 1, 1), block=(64, 1, 1))


@flyc.kernel
def outer_call_only(X: fx.Tensor, Y: fx.Tensor):
    inner(X, Y)                      # 只调用、不 launch → 什么都不生成，outer_call_only 是空 kernel


print("outer_launch:")
try_(lambda: dumps(launcher(outer_launch), X, Y, stream))
print("outer_call_only:")
ir_ = clean(dumps(launcher(outer_call_only), X, Y, stream)["00_origin.mlir"])
body = re.search(r"gpu\.func @\w+\(.*?\{\n(.*?)\s*gpu\.return", ir_, re.S).group(1).strip()
print("    gpu.func:", re.findall(r"gpu\.func @(\w+)", ir_), "| 函数体:", body or "（空）")

outer_launch:
[flydsl.compile] FLYDSL_DUMP_IR=1 dir=/tmp/kernel101_c_0z27q8/launch
[flydsl.compile] dump 00_origin -> /tmp/kernel101_c_0z27q8/launch/00_origin.mlir
[flydsl.compile] dump 01_fly_rewrite_func_signature -> /tmp/kernel101_c_0z27q8/launch/01_fly_rewrite_func_signature.mlir
[flydsl.compile] dump 02_fly_canonicalize -> /tmp/kernel101_c_0z27q8/launch/02_fly_canonicalize.mlir
[flydsl.compile] dump 03_fly_rocdl_expand_ops -> /tmp/kernel101_c_0z27q8/launch/03_fly_rocdl_expand_ops.mlir
[flydsl.compile] dump 04_fly_layout_lowering -> /tmp/kernel101_c_0z27q8/launch/04_fly_layout_lowering.mlir
[flydsl.compile] dump 05_fly_int_swizzle_simplify -> /tmp/kernel101_c_0z27q8/launch/05_fly_int_swizzle_simplify.mlir
[flydsl.compile] dump 06_canonicalize -> /tmp/kernel101_c_0z27q8/launch/06_canonicalize.mlir
[flydsl.compile] dump 07_fly_convert_atom_call_to_ssa_form -> /tmp/kernel101_c_0z27q8/launch/07_fly_convert_atom_call_to_ssa_form.mlir
[flydsl.compile] dump 08_fly_promote_regmem_to_vector

- 想在 kernel 之间复用代码，把公共部分写成 `@flyc.jit` 子函数或普通函数，两个 kernel 各自调用。

## 7. 推荐的组织方式

把上面的规则合起来，一个 kernel 文件通常是这样：

```python
import functools
import flydsl.compiler as flyc
import flydsl.expr as fx


def tile_offset(r, c, BK):            # 只有编译期逻辑 / 地址计算：普通函数
    return r * BK + c


@flyc.jit
def reduce_row(v):                   # 有运行时 if / for：@flyc.jit 子函数（内联）
    ...


@functools.lru_cache                 # 每组编译期参数只 build 一次
def build_gemm(M, N, K, BM, BN, BK, dtype):
    @fx.struct
    class Smem:                      # LDS 结构也可以依赖编译期参数
        a: fx.Array[dtype, BM * BK, 16]

    @flyc.kernel(known_block_size=(256, 1, 1))
    def gemm(A: fx.Tensor, B: fx.Tensor, C: fx.Tensor):
        for kt in range(fx.Int32(0), fx.Int32(K // BK), fx.Int32(1), init=[...]):   # 运行时主循环
            for i in fx.range_constexpr(BM // 16):                                      # 展开的内层
                ...
            results = yield [...]

    @flyc.jit
    def launch(A: fx.Tensor, B: fx.Tensor, C: fx.Tensor, stream: fx.Stream):
        gemm(A, B, C).launch(grid=(M // BM, N // BN, 1), block=(256, 1, 1), stream=stream)

    return launch


launch = build_gemm(4096, 4096, 4096, 256, 256, 64, fx.BFloat16)
compiled = flyc.compile(launch, A, B, C, stream)    # 编译一次（也会运行一次）
for _ in range(iters):
    compiled(A, B, C, stream)                       # 热循环里不再查缓存
```

## 总结

| 问题 | 结论（FlyDSL 0.3.4.1） |
|---|---|
| kernel 和 jit | `@flyc.kernel` → `gpu.func`（每组 `Constexpr` 值一个特化，名字带 `_0`、`_1`）；`@flyc.jit` → host 端 `func.func`，负责算 grid 和 `gpu.launch_func`。kernel 只能在 jit 的 trace 里被调用 |
| 什么时候编译 | jit 函数第一次被调用时 trace + 编译；缓存命中时不再执行 Python 函数体。key = 参数签名（dtype / rank / 静态 layout / `Constexpr` 值）+ 目标 + 环境 + 用到的全局变量。shape 是运行时值，换 shape 不重编 |
| 反复 launch | `flyc.compile(...)` 返回 `CompiledFunction`，跳过参数绑定和缓存查找 |
| 全局变量 | 编译后被修改直接报错 |
| `build(...)` + `lru_cache` | 编译期参数通过闭包传入，每组参数一组独立的 kernel / jit 对象；`lru_cache` 让同一组参数复用同一个 launcher（不加则每次重新编译）。编译目标在 launcher 第一次调用时固定 |
| AST 改写 | `for range` / `if` / 三元 / `and or not` / `while` 改写成对 `scf_*_dispatch` 的调用，分支和循环体变成嵌套函数；运行时条件生成 `scf.if` / `scf.for` / `scf.while`，Python 条件直接执行 |
| 改写的副作用 | 分支里新定义的变量带不出来（`NameError`，Python 条件也一样，除非用 `fx.const_expr`）；`None` 不能做初值；运行时 `if` 里的 `return` 被静默忽略；循环体里不能 `break` |
| for 循环 | `range_constexpr` trace 时展开；`range(常数)` 是 `scf.for`，LLVM 完全展开；`range(运行时)` 是真循环（默认按 4 展开）；`unroll=` 给展开提示；`init=[...]` + `yield` 显式指定 loop-carried 值 |
| 子函数 | 普通函数不改写，只能用编译期控制流（相当于宏）；`@flyc.jit` 子函数或 kernel 内的嵌套函数会改写并内联，可以用运行时 `if` / `for` |
| kernel 调 kernel | `.launch()` 会在 `gpu.func` 里发出 `gpu.launch_func`，无法 lower；不 launch 则什么都不生成。复用代码用子函数 |